In [ ]:
import os
import time
import math
import json
import random
import pandas as pd
import numpy as np
import scipy as sp
import pylab as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
import jupyter_black

jupyter_black.load()

In [ ]:
plt.style.use("../src/mpl_style.txt")

### Excess word list with frequency cutoff (non-lemmatized)
To determine the optimal rare word list that yields the best LLM use estimate,
we try different frequency cutoffs. The base list consists of all excess style 
words from the 2024 pubmed data. For each cutoff value, only words with frequency 
lower than the cutoff (average 2024) are considered into the list 
of rare words. 

In this version, word lemmata are being ignored and each 
version of a word is treated as a separate entity, e.g "delves" != "delve"
This could be a meaningful difference, because LLMs might have preferences
for certain versions of a word, but not others (how does tokenization work 
for ChatGPT? is the base token for "delve", "delves", "delving" the same?
is this relevant? despite having the same base tokens, different variants
occurr in different semantic and syntactic contexts, so to differentiate
between variants could still be informative, despite being based on the 
same token)

Procedure:
- get all 2024 excess words from the previous paper 
- compute the non-lemmatized average 2024 frequencies for the excess words 
(if they are present in the section vocabulary)
- filter list to only include words with average 2024/2025 frequency below cutoff
- group counts for all words in list
- use new counts to compute frequency, diffs, ratios and projection 
- repeat for every cutoff and section

Open questions:
- should there be a lower bound for frequencies (i.e. 1e-4) as there was in the 
paper? The excess words were determined with a lower bound, so they already 
exclude artefacts

In [ ]:
VERSION = "research-article_aimrd_f_crop255"
RESULTS_PATH = "../data/results/baseline_2025-06-26/" + VERSION
secs = next(os.walk(RESULTS_PATH))[1]
# values are either "" or a section name, if not "", only use frequs of the given
# section to compute excess word lists to be used for all sections
vocab_version = "full"

In [ ]:
years = np.arange(2000, 2026)
months = np.arange(1, 13)
with open(os.path.join(RESULTS_PATH, "filters.json")) as f:
    filters = json.load(f)
dates = pd.to_datetime(filters["all_dates"])

In [ ]:
excess_words = utils.excess_style_words_2024
len(excess_words)

In [ ]:
cutoffs = [0.0002, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
excess_words_cutoff = {}
cutoff_counts = np.zeros((len(secs), len(cutoffs)), dtype=int)

for i, sec in enumerate(secs):
    if not vocab_version == "":
        if not sec == vocab_version:
            continue

    excess_words_cutoff[sec] = {}

    X = sp.sparse.load_npz(os.path.join(RESULTS_PATH, sec, f"count_{sec}.pkl.npz"))
    words = np.load(
        os.path.join(RESULTS_PATH, sec, f"words_{sec}.pkl.npy"), allow_pickle=True
    )

    ind_words = np.isin(words, excess_words)
    ind = dates.year == 2024

    counts = np.array(np.sum(X[ind, :][:, ind_words], axis=0)).ravel()
    totals = np.sum(ind)

    freqs_2024 = (counts + 1) / (totals + 1)

    for j, cutoff in enumerate(cutoffs):

        ind_excess = freqs_2024 < cutoff

        cutoff_counts[i, j] = np.sum(ind_excess)
        excess_words_cutoff[sec][cutoff] = words[ind_words][ind_excess]

pd.DataFrame(cutoff_counts, columns=cutoffs, index=secs)

In [ ]:
excess_words_cutoff.keys()

In [ ]:
freqs_dfs = {}

for sec in secs:
    if vocab_version == "":
        freqs_dfs[sec] = utils.get_cutoff_frequency(
            RESULTS_PATH,
            sec,
            excess_words_cutoff[sec],
            lemmatized="_non-lemmatized",
        )
    else:
        freqs_dfs[sec] = utils.get_cutoff_frequency(
            RESULTS_PATH,
            sec,
            excess_words_cutoff[vocab_version],
            lemmatized=f"_non-lemmatized_{vocab_version}_only",
        )

In [ ]:
# filter out cutoffs with too high base frequency
freqs_dfs_filtered = {}
for sec in secs:
    remove = freqs_dfs[sec][freqs_dfs[sec]["projection"] >= 0.99]["cutoff"].unique()
    freqs_dfs_filtered[sec] = freqs_dfs[sec][
        freqs_dfs[sec]["cutoff"].apply(lambda x: x not in remove)
    ].copy()

In [ ]:
freqs_dfs["introduction"]

In [ ]:
freqs_dfs_filtered["introduction"]

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(5, 5), layout="constrained")

hue_norm = LogNorm()
palette = "flare"
data = freqs_dfs_filtered["results"]
for i, var in enumerate(["frequency", "diff", "usage estimate"]):
    ax = axs.flatten()[i]

    ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
    sns.lineplot(
        data=data,
        x="time",
        y=var,
        hue="cutoff",
        hue_norm=hue_norm,
        palette=palette,
        ax=ax,
        legend=False,
    )

    if var == "frequency":
        sns.lineplot(
            data=data,
            x="time",
            y="projection",
            hue="cutoff",
            hue_norm=LogNorm(),
            palette=palette,
            ax=ax,
            alpha=0.5,
            linestyle="-.",
            legend=False,
        )

    if var == "usage estimate":
        ax.set_ylim(-0.01, 1.01)

    if var == "diff":
        sm = plt.cm.ScalarMappable(cmap=palette, norm=hue_norm)
        cax = fig.add_axes(
            [
                ax.get_position().x1 + 0.1,
                ax.get_position().y0,
                0.02,
                ax.get_position().height * 2,
            ]
        )
        ax.figure.colorbar(sm, cax=cax)

In [ ]:
freqs_df_agg = {}
for sec in secs:
    df = freqs_dfs[sec].copy()
    df["time"] = list(map(math.floor, df["time"]))
    df = df.groupby(["time", "cutoff"]).mean().reset_index()
    freqs_df_agg[sec] = df

In [ ]:
freqs_df_filtered_agg = {}
for sec in secs:
    df = freqs_dfs_filtered[sec].copy()
    df["time"] = list(map(math.floor, df["time"]))
    df = df.groupby(["time", "cutoff"]).mean().reset_index()
    freqs_df_filtered_agg[sec] = df

In [ ]:
freqs_df_agg["introduction"][freqs_df_agg["introduction"]["time"] == 2024]

In [ ]:
# ensure correct order
sections = ["abstract", "introduction", "methods", "results", "discussion", "full"]

fig = plt.figure(figsize=(7, 6), layout="constrained")
subfigs = fig.subfigures(3, 2)  # outer grid

for subfig, sec in zip(subfigs.flat, sections):
    f = freqs_df_agg[sec][freqs_df_agg[sec]["time"] == 2025]
    f_filt = freqs_df_filtered_agg[sec][freqs_df_filtered_agg[sec]["time"] == 2025]

    axs = subfig.subplots(1, 2)  # inner grid

    subfig.suptitle(sec)

    # frequency plot
    axs[0].plot(
        f["cutoff"], f["frequency"], ".-", clip_on=False, label="Observed\nfrequency"
    )
    axs[0].plot(
        f["cutoff"], f["projection"], ".-", clip_on=False, label="Expected\nfrequency"
    )
    axs[0].set_xscale("log")
    axs[0].set_xlim([1e-4, 1])
    axs[0].set_ylim([0, 1])
    axs[0].set_ylabel("Frequency")
    axs[0].set_xlabel("Threshold")
    if sec == "abstract":
        axs[0].legend()

    # usage estimate plot
    axs[1].plot(f["cutoff"], f["diff"], "k.-", label="Δ")
    axs[1].plot(f_filt["cutoff"], f_filt["usage estimate"], "r.-", label="Δ / (1-Proj)")

    i = int(np.argmax(f_filt["usage estimate"]))
    axs[1].text(
        list(f_filt["cutoff"])[i] + 0.04,
        list(f_filt["usage estimate"])[i] + 0.04,
        f"{list(f["usage estimate"])[i]:.2f}",
        va="center",
        bbox=dict(
            edgecolor="none", facecolor="#666666", alpha=0.5, boxstyle="round,pad=.2"
        ),
    )
    i = int(np.argmax(f["diff"]))
    axs[1].text(
        list(f["cutoff"])[i] + 0.01,
        list(f["diff"])[i] + 0.04,
        f"{list(f["diff"])[i]:.2f}",
        va="center",
        bbox=dict(
            edgecolor="none", facecolor="#666666", alpha=0.5, boxstyle="round,pad=.2"
        ),
    )
    axs[1].set_xscale("log")
    axs[1].set_xlim([1e-4, 1.15])
    axs[1].set_ylim([0, 0.8])
    axs[1].set_ylabel(r"Frequency gap ($\Delta$)")
    axs[1].set_xlabel("Threshold")
    if sec == "abstract":
        axs[1].legend()

plt.savefig(
    os.path.join(
        "../results/plots", VERSION, f"lower_bound_non_lemmatized_{vocab_version}.png"
    ),
    dpi=300,
)

In [ ]:
np.argmax(f["usage estimate"])

In [ ]:
f["usage estimate"]

In [ ]:
excess_words_cutoff["introduction"][0.05]